# 18 - Gemini Annotation Datasets

Reuters ve SP500 verilerini Gemini ile etiketler.

**Çalışma mantığı**
- Gemini/API hatası olursa program DURMAZ.
- Hatalı kayıt başarılı cevap gelene kadar otomatik olarak yeniden denenir.
- Hata sonrası bekleme: 5 → 10 → 20 → 40 → 80 → 120 sn (120 sn'de sabitlenir).
- 429 (rate limit) hatasında API'nin önerdiği `retryDelay` değeri de dikkate alınır.
- İstekler arası minimum `REQUEST_INTERVAL` saniye beklenir (free tier 5 RPM için 13 sn).
- Geçerli Gemini cevabı gelir gelmez `gemini_label` yazılır.
- **Her başarılı kayıt hemen checkpoint'e kaydedilir.**
- Reuters ve SP500 checkpoint dosyaları birbirinden ayrıdır.
- Notebook yeniden çalıştırılırsa başarılı kayıtlar korunur ve kalan kayıtlar kaldığı yerden devam eder.
- Checkpoint klasörü: `db/annotations/gemini/checkpoints/`


# 1. Kurulum

Gerekli paketler.


In [1]:
%pip install -q -U google-genai openpyxl pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Import ve genel ayarlar


In [2]:
from pathlib import Path
import os
import time
import re
from getpass import getpass

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# 3. Proje yolları


In [3]:
NOTEBOOK_DIR = Path.cwd()

PROJECT_DIR = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == "app"
    else NOTEBOOK_DIR
)

DATA_DIR = PROJECT_DIR / "db"
annotations_dir = DATA_DIR / "annotations"

print("=" * 100)
print("PROJE YOLLARI")
print("=" * 100)
print(f"PROJECT_DIR : {PROJECT_DIR}")
print(f"DATA_DIR    : {DATA_DIR}")
print(f"ANNOTATIONS : {annotations_dir}")

PROJE YOLLARI
PROJECT_DIR : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis
DATA_DIR    : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db
ANNOTATIONS : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations


# 4. Reuters veri setini yükle


In [4]:
reuters_path = (
    annotations_dir
    / "reuters_5000"
    / "REUTERS_annotation_all_clean.csv"
)

if not reuters_path.exists():
    raise FileNotFoundError(
        f"Reuters dosyası bulunamadı:\n{reuters_path}"
    )

reuters_df = pd.read_csv(
    reuters_path,
    encoding="utf-8-sig"
)

print("=" * 100)
print("REUTERS")
print("=" * 100)
print(f"Dosya : {reuters_path}")
print(f"Satır : {len(reuters_df)}")
print(f"Kolon : {len(reuters_df.columns)}")
print("\nKolonlar:")
print(list(reuters_df.columns))

REUTERS
Dosya : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\REUTERS_annotation_all_clean.csv
Satır : 5000
Kolon : 18

Kolonlar:
['annotation_id', 'date', 'text_en', 'text_tr', 'finbert_score', 'finbert_label', 'finbert_confidence', 'chatgpt_label', 'chatgpt_score', 'chatgpt_confidence', 'chatgpt_reason_tr', 'final_label', 'finbert_correctness', 'review_needed', 'row_note', 'batch_no', 'n_words', 'text_len']


# 5. SP500 batch dosyalarını yükle


In [5]:
sp500_batches_dir = (
    annotations_dir
    / "sp500_human_review_batches"
)

if not sp500_batches_dir.exists():
    raise FileNotFoundError(
        f"SP500 klasörü bulunamadı:\n{sp500_batches_dir}"
    )

sp500_dfs = []

for batch_file in sorted(
    sp500_batches_dir.glob("SP500_annotation_batch_*.xlsx")
):
    try:
        batch_df = pd.read_excel(
            batch_file,
            engine="openpyxl"
        )
        sp500_dfs.append(batch_df)

        print(
            f"✓ {batch_file.name} "
            f"| {len(batch_df)} satır "
            f"| {len(batch_df.columns)} kolon"
        )

    except Exception as e:
        raise RuntimeError(
            f"SP500 dosyası okunamadı:\n{batch_file}\n\nHata: {e}"
        ) from e

if not sp500_dfs:
    raise RuntimeError(
        "Hiçbir SP500 batch dosyası okunamadı."
    )

sp500_combined = pd.concat(
    sp500_dfs,
    ignore_index=True
)

print("\n" + "=" * 100)
print("SP500 BİRLEŞTİRİLMİŞ")
print("=" * 100)
print(f"Satır : {len(sp500_combined)}")
print(f"Kolon : {len(sp500_combined.columns)}")
print("\nKolonlar:")
print(list(sp500_combined.columns))

✓ SP500_annotation_batch_001.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_002.xlsx | 10 satır | 17 kolon
✓ SP500_annotation_batch_004.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_005.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_006.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_007.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_008.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_009.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_010.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_011.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_012.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_013.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_014.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_015.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_016.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_017.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_018.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_019.xlsx | 10 satır | 1

# 6. Reuters'i 3'er sütun halinde incele

Bu hücre sadece veri setini analiz eder. **Henüz Gemini çağrısı yapmaz.**


In [6]:
print("=" * 100)
print("REUTERS - 3'ER SÜTUN")
print("=" * 100)
print(f"Satır: {len(reuters_df)} | Kolon: {len(reuters_df.columns)}")

for i in range(0, len(reuters_df.columns), 3):
    print(f"\nKolonlar: {list(reuters_df.columns[i:i+3])}")
    display(
        reuters_df.iloc[:, i:i+3].head(5)
    )


REUTERS - 3'ER SÜTUN
Satır: 5000 | Kolon: 18

Kolonlar: ['annotation_id', 'date', 'text_en']


,annotation_id,date,text_en
0,REUTERS_ANN_00001,2007-04-12,s.africa watchdog to probe gold fields bid report
1,REUTERS_ANN_00002,2007-04-26,new barbie girls sashay into view with mp-3
2,REUTERS_ANN_00003,2006-12-17,"stocks await data, mergers and santa"
3,REUTERS_ANN_00004,2007-01-03,canadian dollar falls to 9-mth lows vs u.s. dollar
4,REUTERS_ANN_00005,2007-02-05,"techs buckle under microsoft, wal-mart helps dow"



Kolonlar: ['text_tr', 'finbert_score', 'finbert_label']


,text_tr,finbert_score,finbert_label
0,Güney Afrika rekabet kurumu Gold Fields teklifi raporunu inceleyecek,-1,negative
1,Yeni Barbie Girls MP3 ile sahneye çıkıyor,0,neutral
2,"Hisseler veri, birleşmeler ve Noel Baba rallisini bekliyor",0,neutral
3,Kanada doları ABD doları karşısında 9 ayın en düşük seviyesine düştü,-1,negative
4,"Teknoloji hisseleri Microsoft baskısıyla çöktü, Wal-Mart Dow’a yardım etti",-1,negative



Kolonlar: ['finbert_confidence', 'chatgpt_label', 'chatgpt_score']


...

,chatgpt_confidence,chatgpt_reason_tr,final_label
0,0.86,Bir satın alma teklifinin rekabet kurumu tarafından incelenmesi düzenleyici belirsizlik ve işlem riski yaratır.,negative
1,0.76,Yeni MP3 özellikli ürün lansmanı ürün yeniliği ve satış potansiyeli açısından sınırlı olumlu sinyal verir.,positive
2,0.82,Piyasa bekleyişi anlatılıyor; veri veya birleşmelerin net sonucu başlıkta yok.,neutral
3,0.98,Para biriminin 9 ayın en düşük seviyesine gerilemesi güçlü negatif döviz hareketidir.,negative
4,0.86,Wal-Mart desteği olumlu olsa da teknoloji hisselerindeki baskı ve 'buckle' ifadesi baskın negatif piyasa sinyali verir.,negative



Kolonlar: ['finbert_correctness', 'review_needed', 'row_note']


,finbert_correctness,review_needed,row_note
0,same,no,FinBERT etiketi bağlamla uyumlu; 'watchdog to probe' ifadesi düzenleyici soruşturma/inceleme riskini negatif yakalamış.
1,different,yes,FinBERT bunu nötr görmüş olabilir çünkü başlık finansal metrik içermiyor; ancak yeni ürün tanıtımı pozitif ürün/growth sinyalidir.
2,same,no,FinBERT etiketi bağlamla uyumlu; 'await' bekle-gör ve yönsüz piyasa tonudur.
3,same,no,FinBERT etiketi bağlamla uyumlu; 'falls to lows' net negatif.
4,same,no,FinBERT etiketi bağlamla uyumlu; tech weakness başlığın negatif tarafını yakalamış.



Kolonlar: ['batch_no', 'n_words', 'text_len']


,batch_no,n_words,text_len
0,1,8,49
1,1,8,43
2,1,6,36
3,1,9,50
4,1,7,48


# 7. SP500'ü 3'er sütun halinde incele


In [7]:
print("=" * 100)
print("SP500 - 3'ER SÜTUN")
print("=" * 100)
print(f"Satır: {len(sp500_combined)} | Kolon: {len(sp500_combined.columns)}")

for i in range(0, len(sp500_combined.columns), 3):
    print(f"\nKolonlar: {list(sp500_combined.columns[i:i+3])}")
    display(
        sp500_combined.iloc[:, i:i+3].head(5)
    )


SP500 - 3'ER SÜTUN
Satır: 1067 | Kolon: 32

Kolonlar: ['annotation_id', 'sample_id', 'date']


,annotation_id,sample_id,date
0,SP500_ANN_0001,SP500_HEAD_000004,2008-01-03
1,SP500_ANN_0002,SP500_HEAD_016918,2023-12-15
2,SP500_ANN_0003,SP500_HEAD_013289,2023-03-20
3,SP500_ANN_0004,SP500_HEAD_017167,2024-01-04
4,SP500_ANN_0005,SP500_HEAD_007040,2019-09-18



Kolonlar: ['text_en', 'text_tr', 'finbert_label']


,text_en,text_tr,finbert_label
0,"U.S. Stocks Higher After Economic Data, Monsanto Outlook",ABD hisseleri ekonomik veriler ve Monsanto görünümü sonrası yükseldi.,positive
1,Stock Market Outlook 2024: Rare Bullish Signal Says S&P 500 Will Soar 20%,2024 borsa görünümü: Nadir bir boğa sinyali S&P 500'ün %20 yükseleceğini söylüyor.,positive
2,Federal Reserve Rate Hike Odds Grow As Bank-Crisis Fears Ebb; S&P 500 Rises,Banka krizi korkuları azalırken Fed faiz artırımı olasılığı yükseliyor; S&P 500 yükseliyor.,negative
3,"US Stock Market Closing: Dow Jones, S&P 500 Retreat As Traders Assess Fed Meeting Minutes, Nasdaq Plunges 1.2","ABD borsa kapanışı: Fed tutanakları değerlendirilirken Dow Jones ve S&P 500 geriledi, Nasdaq %1,2 düştü.",negative
4,"Deutsche Bank: S&P 500 13% Overvalued, Recession Coming","Deutsche Bank: S&P 500 %13 aşırı değerli, resesyon geliyor.",negative



Kolonlar: ['finbert_confidence', 'chatgpt_label', 'chatgpt_confidence']


...

,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN



Kolonlar: ['Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']


,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN



Kolonlar: ['SP500 Annotation Batch 107', 'SP500 Annotation Batch 109']


,SP500 Annotation Batch 107,SP500 Annotation Batch 109
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN


# 8. Veri seti kalite ve etiket analizi


In [8]:
print("=" * 100)
print("VERİ SETİ ANALİZİ")
print("=" * 100)

print("\nREUTERS BOŞ DEĞERLER:")
print(reuters_df.isna().sum())

print("\nREUTERS CHATGPT LABEL DAĞILIMI:")
if "chatgpt_label" in reuters_df.columns:
    print(reuters_df["chatgpt_label"].value_counts(dropna=False))
else:
    print("chatgpt_label kolonu yok.")

print("\nSP500 BOŞ DEĞERLER:")
print(sp500_combined.isna().sum())

print("\nSP500 CHATGPT LABEL DAĞILIMI:")
if "chatgpt_label" in sp500_combined.columns:
    print(sp500_combined["chatgpt_label"].value_counts(dropna=False))
else:
    print("chatgpt_label kolonu yok.")

print("\nTEXT KOLON KONTROLÜ:")
print("Reuters text_en var mı? ->", "text_en" in reuters_df.columns)
print("SP500 text_en var mı?  ->", "text_en" in sp500_combined.columns)

VERİ SETİ ANALİZİ

REUTERS BOŞ DEĞERLER:
annotation_id             0
date                      0
text_en                   0
text_tr                   0
finbert_score             0
finbert_label             0
finbert_confidence        0
chatgpt_label             0
chatgpt_score           350
chatgpt_confidence        0
chatgpt_reason_tr         0
final_label               0
finbert_correctness    1300
review_needed          1454
row_note               1856
batch_no                  0
n_words                   0
text_len                  0
dtype: int64

REUTERS CHATGPT LABEL DAĞILIMI:
chatgpt_label
positive    2275
negative    1579
neutral     1146
Name: count, dtype: int64

SP500 BOŞ DEĞERLER:
annotation_id                   37
sample_id                       37
date                            37
text_en                         37
text_tr                         37
finbert_label                   37
finbert_confidence              37
chatgpt_label                   37
chatgpt_confidenc

# 9. Gemini için kullanılacak aday veri setini hazırla

Bu hücre de sadece hazırlık yapar; API çağrısı yoktur.


In [9]:
datasets_for_gemini = []

# SP500
if (
    not sp500_combined.empty
    and "text_en" in sp500_combined.columns
    and "chatgpt_label" in sp500_combined.columns
):
    sp500_cols = [
        c
        for c in [
            "annotation_id",
            "sample_id",
            "date",
            "text_en",
            "chatgpt_label",
        ]
        if c in sp500_combined.columns
    ]

    sp500_cleaned = (
        sp500_combined[sp500_cols]
        .copy()
    )

    sp500_cleaned = sp500_cleaned[
        sp500_cleaned["text_en"].notna()
        &
        sp500_cleaned["chatgpt_label"].notna()
    ].copy()

    sp500_gemini = pd.DataFrame({
        "source_dataset": "sp500_human_review",
        "source_row_index": range(len(sp500_cleaned)),
        "annotation_id": (
            sp500_cleaned["annotation_id"]
            .astype(str)
            .str.strip()
            .values
        ),
        "text": (
            sp500_cleaned["text_en"]
            .astype(str)
            .str.strip()
            .values
        ),
        "chatgpt_label": (
            sp500_cleaned["chatgpt_label"]
            .astype(str)
            .str.strip()
            .str.lower()
            .values
        ),
    })

    sp500_gemini = sp500_gemini[
        sp500_gemini["text"].str.len() > 0
    ].copy()

    datasets_for_gemini.append(sp500_gemini)

# Reuters
reuters_gemini = pd.DataFrame({
    "source_dataset": "reuters_annotation_clean",
    "source_row_index": reuters_df.index,
    "annotation_id": (
        reuters_df["annotation_id"]
        .astype(str)
        .str.strip()
        .values
    ),
    "text": (
        reuters_df["text_en"]
        .astype(str)
        .str.strip()
        .values
    ),
    "chatgpt_label": (
        reuters_df["chatgpt_label"]
        .astype(str)
        .str.strip()
        .str.lower()
        .values
    ),
})

reuters_gemini = reuters_gemini[
    reuters_gemini["chatgpt_label"].notna()
    &
    (~reuters_gemini["chatgpt_label"].isin([
        "nan", "none", ""
    ]))
    &
    (reuters_gemini["text"].str.len() > 0)
].copy()

datasets_for_gemini.append(reuters_gemini)

if not datasets_for_gemini:
    raise RuntimeError(
        "Gemini için hiç veri oluşturulamadı."
    )

datasets_for_gemini = pd.concat(
    datasets_for_gemini,
    ignore_index=True
)

datasets_for_gemini["gemini_label"] = pd.NA

print("=" * 100)
print("GEMINI ADAY DATASET")
print("=" * 100)
print(f"Toplam satır : {len(datasets_for_gemini)}")

print("\nDataset dağılımı:")
print(
    datasets_for_gemini["source_dataset"]
    .value_counts()
)

print("\nChatGPT label dağılımı:")
print(
    datasets_for_gemini["chatgpt_label"]
    .value_counts(dropna=False)
)

GEMINI ADAY DATASET
Toplam satır : 6030

Dataset dağılımı:
source_dataset
reuters_annotation_clean    5000
sp500_human_review          1030
Name: count, dtype: int64

ChatGPT label dağılımı:
chatgpt_label
positive    2659
negative    1893
neutral     1478
Name: count, dtype: int64


# 10. Gemini aday datasetini yine 3'er sütun halinde kontrol et


In [10]:
print("=" * 100)
print("GEMINI ADAY DATASET - 3'ER SÜTUN")
print("=" * 100)
print(f"Satır: {len(datasets_for_gemini)} | Kolon: {len(datasets_for_gemini.columns)}")

for i in range(0, len(datasets_for_gemini.columns), 3):
    print(f"\nKolonlar: {list(datasets_for_gemini.columns[i:i+3])}")
    display(
        datasets_for_gemini.iloc[:, i:i+3].head(5)
    )


GEMINI ADAY DATASET - 3'ER SÜTUN
Satır: 6030 | Kolon: 6

Kolonlar: ['source_dataset', 'source_row_index', 'annotation_id']


,source_dataset,source_row_index,annotation_id
0,sp500_human_review,0,SP500_ANN_0001
1,sp500_human_review,1,SP500_ANN_0002
2,sp500_human_review,2,SP500_ANN_0003
3,sp500_human_review,3,SP500_ANN_0004
4,sp500_human_review,4,SP500_ANN_0005



Kolonlar: ['text', 'chatgpt_label', 'gemini_label']


,text,chatgpt_label,gemini_label
0,"U.S. Stocks Higher After Economic Data, Monsanto Outlook",positive,<NA>
1,Stock Market Outlook 2024: Rare Bullish Signal Says S&P 500 Will Soar 20%,positive,<NA>
2,Federal Reserve Rate Hike Odds Grow As Bank-Crisis Fears Ebb; S&P 500 Rises,positive,<NA>
3,"US Stock Market Closing: Dow Jones, S&P 500 Retreat As Traders Assess Fed Meeting Minutes, Nasdaq Plunges 1.2",negative,<NA>
4,"Deutsche Bank: S&P 500 13% Overvalued, Recession Coming",negative,<NA>


# 11. Gemini API ayarları

Bu hücrede API bağlantısı hazırlanır. Etiketleme henüz başlamaz.


In [ ]:
from pathlib import Path
import os
import time
import re
from getpass import getpass

from google import genai
from google.genai import types

# API Key
api_key = ""
# REST + 15 sn timeout
client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(timeout=15000),
)

MODEL_NAME = "gemini-3.5-flash-lite"
REQUEST_INTERVAL = 13.0

SENTIMENT_LABELS = {
    "positive",
    "negative",
    "neutral",
}

PROMPT_TEMPLATE = '''
You are an expert in financial sentiment analysis.

Classify the sentiment of the following financial text.

Text:
{text}

Choose exactly ONE label:

positive
negative
neutral

Definitions:
- positive: favorable financial/business outlook, gains, growth, improvement,
  strong performance, bullish news.
- negative: unfavorable financial/business outlook, losses, decline,
  deterioration, weak performance, bearish news.
- neutral: factual or informational statement without a clearly positive
  or negative financial sentiment.

IMPORTANT:
Return ONLY ONE WORD:
positive
negative
neutral
'''

print("=" * 100)
print("GEMINI API AYARLARI")
print("=" * 100)
print(f"MODEL            : {MODEL_NAME}")
print(f"REQUEST_INTERVAL : {REQUEST_INTERVAL} sn ({60 / REQUEST_INTERVAL:.2f} RPM)")
print("=" * 100)

sample_text = "Revenue for the quarter increased by 15% compared to last year."
formatted_prompt = PROMPT_TEMPLATE.format(text=sample_text)

print("İstek atılıyor...")
start = time.time()

try:
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=formatted_prompt,
        config=types.GenerateContentConfig(
            temperature=0.0,
            # http_options BURADA DEĞİL — Client'a taşındı
        ),
    )
    print(f"Tamamlandı ({time.time() - start:.2f} saniye):", response.text.strip())
except Exception as e:
    print(f"Hata veya Zaman Aşımı Yapıldı ({time.time() - start:.2f} saniye):", e)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


GEMINI API AYARLARI
MODEL            : gemini-3.5-flash-lite
REQUEST_INTERVAL : 13.0 sn (4.62 RPM)
İstek atılıyor...
Tamamlandı (0.98 saniye): positive


# 12. Label yardımcı fonksiyonları


In [12]:
def extract_response_text(response):
    """Gemini cevabından metni güvenli şekilde çıkarır."""
    # Yol 1: response.text (SDK kolaylığı)
    try:
        text = getattr(response, "text", None)
        if text:
            return str(text)
    except Exception:
        pass

    # Yol 2: candidates -> content -> parts -> text
    try:
        for candidate in (getattr(response, "candidates", None) or []):
            content = getattr(candidate, "content", None)
            if content is None:
                continue
            for part in (getattr(content, "parts", None) or []):
                part_text = getattr(part, "text", None)
                if part_text:
                    return str(part_text)
    except Exception:
        pass

    return ""


def normalize_label(raw_text):
    """Ham metni positive/negative/neutral'den birine indirger."""
    if raw_text is None:
        return None

    text = str(raw_text).strip().lower()
    cleaned = re.sub(r"[^a-z]", "", text)

    # Direkt eşleşme
    if cleaned in SENTIMENT_LABELS:
        return cleaned

    # Metnin içinde geçiyor mu?
    for lab in ("positive", "negative", "neutral"):
        if lab in cleaned:
            return lab

    return None


def print_full_response(response):
    """Debug için ham cevabı yazdırır."""
    print("=" * 100)
    print("🔎 GEMINI RAW RESPONSE")
    print("=" * 100)
    print("[TYPE]", type(response))
    print("[response.text]", repr(getattr(response, "text", None)))
    try:
        print("[str(response)]", str(response)[:3000])
    except Exception as e:
        print("[str(response)] yazdırılamadı:", e)
    print("=" * 100)


def _extract_retry_delay(err):
    """429 hatasından API'nin önerdiği retryDelay'i (saniye) çekmeyi dener."""
    try:
        msg = str(err)
        m = re.search(r"retryDelay['\"]?\s*[:=]\s*['\"]?(\d+(?:\.\d+)?)s", msg)
        if m:
            return float(m.group(1))
    except Exception:
        pass
    return None


def ask_gemini(text, debug=False):
    """
    Gemini çağrısını başarılı ve geçerli bir sentiment etiketi gelene kadar dener.

    - API/ClientError dahil tüm Exception'lar yakalanır.
    - Hata notebook hücresine traceback olarak taşınmaz.
    - Bekleme: 5, 10, 20, 40, 80, 120 sn (120 sn'de sabitlenir).
    - 429 (rate limit) hatasında API'nin önerdiği `retryDelay` değeri de dikkate alınır.
    - Geçerli cevap geldiğinde sadece label döner.
    - Ctrl+C / KeyboardInterrupt kullanıcı tarafından durdurma olarak bırakılır.
    """
    prompt = PROMPT_TEMPLATE.format(text=str(text))
    retry_count = 0

    while True:
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    candidate_count=1,
                    max_output_tokens=128,
                    thinking_config=types.ThinkingConfig(
                        thinking_level="high"
                    ),
                ),
            )

            if debug:
                print_full_response(response)

            raw_text = extract_response_text(response)

            if not raw_text:
                raise RuntimeError(
                    "Gemini'den çıkarılabilir metin gelmedi."
                )

            label = normalize_label(raw_text)

            if label is None:
                raise RuntimeError(
                    f"Gemini geçerli bir label döndürmedi. Ham cevap: {raw_text!r}"
                )

            # Başarılı cevap: çağırana dön.
            return label

        except KeyboardInterrupt:
            # Kullanıcı Ctrl+C ile durdurabilsin.
            raise

        except Exception as e:
            retry_count += 1

            # Exponential backoff (5, 10, 20, 40, 80, 120, 120, ...)
            wait_seconds = min(5 * (2 ** (retry_count - 1)), 120)

            # 429 ise API'nin önerdiği retryDelay'i de dikkate al.
            suggested = _extract_retry_delay(e)
            if suggested is not None:
                wait_seconds = max(wait_seconds, suggested + 2)

            short_msg = str(e).split("\n")[0][:300]

            print(
                f"\n⚠️ Gemini çağrısı başarısız | "
                f"Retry #{retry_count} | "
                f"{type(e).__name__}: {short_msg}"
            )
            print(
                f"⏳ {wait_seconds:.1f} saniye sonra aynı kayıt tekrar denenecek..."
            )

            time.sleep(wait_seconds)


### Neden bu ayar?

Gemini 3.6 Flash'ta `max_output_tokens` düşünme tokenlarını da kapsar. `20` çok düşük olduğu için model `MAX_TOKENS` ile kesilebiliyordu. Bu sürümde `thinking_level="minimal"` ve `max_output_tokens=128` kullanılıyor.

**Rate limit notu:** Free tier'da `gemini-3.6-flash` için limit **5 RPM**'dir. `REQUEST_INTERVAL=13.0` (≈4.62 RPM) bu limitin altında kalır. Daha yüksek bir tier'a geçtiğinde veya farklı bir model kullandığında bu değeri düşürebilirsin. `ask_gemini` fonksiyonu yine de her ihtimale karşı 429 hatasında otomatik retry yapar.


# 13. API test

Tek bir örnek gönderilir. Bu test başarısızsa etiketleme başlamaz.


In [13]:
TEST_TEXT = (
    "U.S. Stocks Higher After Economic Data, "
    "Monsanto Outlook"
)

print("=" * 100)
print("GEMINI API TEST")
print("=" * 100)

# ask_gemini kendi içinde başarılı cevap gelene kadar retry eder.
test_label = ask_gemini(
    TEST_TEXT,
    debug=True
)

print("\n" + "=" * 100)
print("✅ API TESTİ BAŞARILI")
print("=" * 100)
print(f"Gemini label: {test_label}")


GEMINI API TEST
🔎 GEMINI RAW RESPONSE
[TYPE] <class 'google.genai.types.GenerateContentResponse'>
[response.text] 'positive'
[str(response)] sdk_http_response=HttpResponse(
  headers=<dict len=12>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        text='positive',
        thought_signature=b'\x12\xd8\x04\n\xd5\x04\x01\x11M2\x0f\xae{\xc6j\x0f1!id\x03\xbaY;e\xfd!Y\xba\xc5\xeat\xec\x87u?v\xe4=\xd7\xdfL\xa7ey\x04M\xe9%S,R\x1b\xc1\x7f\xe8Y\x82\xc3\xfd#\tB\x9eQ`\xd77\xd1T\x1ff\xc3Z \x7f\xc6\xc0\x87\xa1\xe6\xac\x15\xc8w\xcc\xc2)\xfd\xc5\x8c)\xfc\x88"\xbaE...'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.5-flash-lite' prompt_feedback=None response_id='VjCoapuuG6SH28oP-qfiqQs' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=1,
  prompt_token_count=138,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
    

# 14. Output / checkpoint yolları

Reuters ve SP500 için ayrı checkpoint ve final dosyaları.


In [14]:
GEMINI_OUTPUT_DIR = DATA_DIR / "annotations" / "gemini"
CHECKPOINT_DIR = GEMINI_OUTPUT_DIR / "checkpoints"

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REUTERS_CHECKPOINT_FILE = CHECKPOINT_DIR / "reuters_gemini_checkpoint.csv"
SP500_CHECKPOINT_FILE = CHECKPOINT_DIR / "sp500_gemini_checkpoint.csv"

REUTERS_FINAL_FILE = GEMINI_OUTPUT_DIR / "reuters_gemini_labeled.csv"
SP500_FINAL_FILE = GEMINI_OUTPUT_DIR / "sp500_gemini_labeled.csv"
COMBINED_FINAL_FILE = GEMINI_OUTPUT_DIR / "gemini_annotation_labeled.csv"

print("CHECKPOINT DIR :", CHECKPOINT_DIR)
print("REUTERS CHECKPOINT:", REUTERS_CHECKPOINT_FILE)
print("SP500 CHECKPOINT  :", SP500_CHECKPOINT_FILE)
print("REUTERS FINAL     :", REUTERS_FINAL_FILE)
print("SP500 FINAL       :", SP500_FINAL_FILE)
print("COMBINED FINAL    :", COMBINED_FINAL_FILE)


CHECKPOINT DIR : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\checkpoints
REUTERS CHECKPOINT: d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\checkpoints\reuters_gemini_checkpoint.csv
SP500 CHECKPOINT  : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\checkpoints\sp500_gemini_checkpoint.csv
REUTERS FINAL     : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\reuters_gemini_labeled.csv
SP500 FINAL       : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\sp500_gemini_labeled.csv
COMBINED FINAL    : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\gemini_annotation_labeled.csv


# 15. Resume: Reuters ve SP500 checkpoint'lerini ayrı yükle

Her dataset kendi checkpoint'inden devam eder.


In [15]:
def load_checkpoint_into_dataset(base_df, checkpoint_file):
    """Checkpoint varsa başarılı Gemini etiketlerini base dataframe'e uygular."""
    result = base_df.copy()

    if not checkpoint_file.exists():
        print(f"Checkpoint yok: {checkpoint_file}")
        return result

    checkpoint = pd.read_csv(
        checkpoint_file,
        encoding="utf-8-sig",
        dtype=str
    )

    required = {
        "source_dataset",
        "source_row_index",
        "annotation_id",
        "text",
        "chatgpt_label",
        "gemini_label",
    }

    missing = required - set(checkpoint.columns)

    if missing:
        print(
            f"⚠️ Geçersiz checkpoint ({checkpoint_file.name}); "
            f"eksik kolonlar: {missing}"
        )
        return result

    checkpoint["annotation_id"] = (
        checkpoint["annotation_id"]
        .astype("string")
        .str.strip()
    )

    successful = checkpoint[
        checkpoint["gemini_label"]
        .astype("string")
        .str.lower()
        .isin(["positive", "negative", "neutral"])
    ].copy()

    label_map = (
        successful[
            ["annotation_id", "gemini_label"]
        ]
        .drop_duplicates("annotation_id", keep="last")
        .set_index("annotation_id")["gemini_label"]
        .to_dict()
    )

    result["annotation_id"] = (
        result["annotation_id"]
        .astype("string")
        .str.strip()
    )

    result["gemini_label"] = (
        result["annotation_id"].map(label_map)
        if label_map
        else pd.NA
    )

    print(
        f"✓ {checkpoint_file.name}: "
        f"{result['gemini_label'].notna().sum()} başarılı etiket geri yüklendi."
    )

    return result


reuters_mask = datasets_for_gemini["source_dataset"] == "reuters_annotation_clean"
sp500_mask = datasets_for_gemini["source_dataset"] == "sp500_human_review"

reuters_dataset = datasets_for_gemini[reuters_mask].copy()
sp500_dataset = datasets_for_gemini[sp500_mask].copy()

reuters_dataset = load_checkpoint_into_dataset(
    reuters_dataset,
    REUTERS_CHECKPOINT_FILE
)

sp500_dataset = load_checkpoint_into_dataset(
    sp500_dataset,
    SP500_CHECKPOINT_FILE
)

datasets_for_gemini = pd.concat(
    [sp500_dataset, reuters_dataset],
    ignore_index=True
)

print(f"Reuters : {len(reuters_dataset)} satır")
print(f"SP500   : {len(sp500_dataset)} satır")


Checkpoint yok: d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\checkpoints\reuters_gemini_checkpoint.csv
✓ sp500_gemini_checkpoint.csv: 7 başarılı etiket geri yüklendi.
Reuters : 5000 satır
SP500   : 1030 satır


# 16. Kaç satır kaldığını hesapla


In [16]:
gemini_normalized = (
    datasets_for_gemini["gemini_label"]
    .astype("string")
    .str.strip()
    .str.lower()
)

unlabeled_mask = (
    gemini_normalized.isna()
    |
    gemini_normalized.isin([
        "",
        "nan",
        "none",
        "<na>",
        "hata",
        "error",
    ])
)

unlabeled_indices = (
    datasets_for_gemini[
        unlabeled_mask
    ]
    .index
    .tolist()
)

total_rows = len(
    datasets_for_gemini
)

already_labeled_count = (
    total_rows
    - len(unlabeled_indices)
)

print("=" * 100)
print("ETİKETLEME DURUMU")
print("=" * 100)
print(f"Toplam satır   : {total_rows}")
print(f"Etiketli       : {already_labeled_count}")
print(f"İşlenecek      : {len(unlabeled_indices)}")

remaining_seconds = (
    len(unlabeled_indices)
    * REQUEST_INTERVAL
)

print(
    f"\nRate ayarı     : "
    f"{60 / REQUEST_INTERVAL:.2f} RPM"
)

print(
    f"Tahmini minimum: "
    f"{remaining_seconds / 3600:.2f} saat"
)

ETİKETLEME DURUMU
Toplam satır   : 6030
Etiketli       : 7
İşlenecek      : 6023

Rate ayarı     : 4.62 RPM
Tahmini minimum: 21.75 saat


# 17. Gemini etiketleme

**Otomatik retry sistemi aktif.**

Bir Gemini/API hatası olduğunda:
- durmaz,
- aynı kayıt tekrar denenir,
- bekleme süresi kademeli artar (429 ise API'nin önerdiği süre de dikkate alınır),
- geçerli cevap gelmeden sonraki kayda geçilmez,
- başarılı cevap gelir gelmez kayıt checkpoint'e yazılır.

**Rate limit:** İstekler arasında `REQUEST_INTERVAL` (şu an 13 sn) beklenir.


In [ ]:
# 17. Gemini etiketleme
#
# ask_gemini() kendi içinde başarılı cevap gelene kadar retry eder.
# Bu hücrede:
#   - istekler arası throttle uygulanır,
#   - yalnızca başarılı cevap geldikten sonra checkpoint yazılır.

def save_dataset_checkpoint(df, dataset_name):
    if dataset_name == "reuters_annotation_clean":
        path = REUTERS_CHECKPOINT_FILE
    elif dataset_name == "sp500_human_review":
        path = SP500_CHECKPOINT_FILE
    else:
        raise ValueError(f"Bilinmeyen dataset: {dataset_name}")

    # Checkpoint yazimi da gecici bir hata olursa tamamlanana kadar denenir.
    save_retry = 0
    while True:
        try:
            df.to_csv(
                path,
                index=False,
                encoding="utf-8-sig"
            )
            break
        except KeyboardInterrupt:
            raise
        except Exception as e:
            save_retry += 1
            wait_seconds = min(5 * (2 ** (save_retry - 1)), 120)
            print(
                f"Checkpoint yazilamadi | Retry #{save_retry}: {e}"
            )
            print(f"{wait_seconds} saniye sonra tekrar denenecek...")
            time.sleep(wait_seconds)
    return path


dataset_jobs = [
    ("sp500_human_review", sp500_dataset),
    ("reuters_annotation_clean", reuters_dataset),
]

for dataset_name, current_df in dataset_jobs:

    current_df = current_df.copy()

    normalized = (
        current_df["gemini_label"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    remaining_indices = current_df.index[
        ~normalized.isin(["positive", "negative", "neutral"])
    ].tolist()

    print("\n" + "#" * 110)
    print(f"BAŞLIYOR: {dataset_name}")
    print(f"Toplam kayıt      : {len(current_df)}")
    print(f"Zaten etiketli    : {len(current_df) - len(remaining_indices)}")
    print(f"Etiketlenecek     : {len(remaining_indices)}")
    print("#" * 110)

    # Throttle için: son isteğin zaman damgası
    last_call_ts = 0.0

    for position, idx in enumerate(remaining_indices, start=1):

        row = current_df.loc[idx]

        annotation_id = str(row["annotation_id"])
        text = str(row["text"])

        # ------------------------------------------------------------------
        # Throttle: istekler arasında minimum REQUEST_INTERVAL saniye bekle
        # ------------------------------------------------------------------
        elapsed = time.time() - last_call_ts
        if elapsed < REQUEST_INTERVAL:
            time.sleep(REQUEST_INTERVAL - elapsed)

        print("\n" + "-" * 110)
        print(
            f"[{position}/{len(remaining_indices)}] "
            f"{dataset_name} | {annotation_id}"
        )

        # ask_gemini burada hata vermez:
        # başarılı cevap gelene kadar kendi içinde bekleyip tekrar dener.
        label = ask_gemini(
            text,
            debug=False
        )

        last_call_ts = time.time()

        # Başarılı cevap
        current_df.loc[idx, "gemini_label"] = label

        print(f"✅ Gemini: {label}")

        # Başarılı kaydı ANINDA checkpoint'e yaz.
        checkpoint_path = save_dataset_checkpoint(
            current_df,
            dataset_name
        )

        print(
            f"💾 Checkpoint kaydedildi: {checkpoint_path}"
        )

        # Global dataframe'i de güncelle
        mask = (
            (datasets_for_gemini["source_dataset"] == dataset_name)
            & (datasets_for_gemini["annotation_id"].astype(str)
               == annotation_id)
        )

        datasets_for_gemini.loc[
            mask,
            "gemini_label"
        ] = label

print("\n✅ TÜM ETİKETLEME TAMAMLANDI.")



##############################################################################################################
BAŞLIYOR: sp500_human_review
Toplam kayıt      : 1030
Zaten etiketli    : 7
Etiketlenecek     : 1023
##############################################################################################################

--------------------------------------------------------------------------------------------------------------
[1/1023] sp500_human_review | SP500_ANN_0008
✅ Gemini: neutral
💾 Checkpoint kaydedildi: d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\checkpoints\sp500_gemini_checkpoint.csv

--------------------------------------------------------------------------------------------------------------
[2/1023] sp500_human_review | SP500_ANN_0009
✅ Gemini: positive
💾 Checkpoint kaydedildi: d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\checkpoints\sp500_gemini_checkpoint.csv

--------

# 18. Final dosyaları ve consensus

Reuters ve SP500 ayrı final dosyalarına kaydedilir.


In [ ]:
reuters_final = datasets_for_gemini[
    datasets_for_gemini["source_dataset"] == "reuters_annotation_clean"
].copy()

sp500_final = datasets_for_gemini[
    datasets_for_gemini["source_dataset"] == "sp500_human_review"
].copy()

reuters_final.to_csv(
    REUTERS_FINAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

sp500_final.to_csv(
    SP500_FINAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

datasets_for_gemini.to_csv(
    COMBINED_FINAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("✅ Reuters final:", REUTERS_FINAL_FILE)
print("✅ SP500 final  :", SP500_FINAL_FILE)
print("✅ Birleşik final:", COMBINED_FINAL_FILE)

valid_labels = {"positive", "negative", "neutral"}

chatgpt_norm = (
    datasets_for_gemini["chatgpt_label"]
    .astype("string")
    .str.lower()
    .str.strip()
)

gemini_norm = (
    datasets_for_gemini["gemini_label"]
    .astype("string")
    .str.lower()
    .str.strip()
)

consensus_mask = (
    chatgpt_norm.isin(valid_labels)
    & gemini_norm.isin(valid_labels)
    & (chatgpt_norm == gemini_norm)
)

consensus_df = datasets_for_gemini[consensus_mask].copy()

print("\n" + "=" * 100)
print("CONSENSUS")
print("=" * 100)
print("Toplam:", len(datasets_for_gemini))
print("Consensus:", len(consensus_df))

if len(datasets_for_gemini) > 0:
    print(
        "Consensus rate: %.2f%%"
        % (100 * len(consensus_df) / len(datasets_for_gemini))
    )


# 19. Consensus dosyalarını kaydet


In [ ]:
consensus_output_dir = (
    DATA_DIR
    / "processed"
    / "gemini_consensus_2024"
)

consensus_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

consensus_results = []

for source_name in (
    datasets_for_gemini[
        "source_dataset"
    ]
    .dropna()
    .unique()
):

    source_consensus = (
        consensus_df[
            consensus_df[
                "source_dataset"
            ] == source_name
        ]
        .copy()
    )

    if source_consensus.empty:
        print(
            f"⚠️ {source_name}: 0 consensus satır"
        )
        continue

    consensus_results.append(
        source_consensus
    )

    output_path = (
        consensus_output_dir
        /
        f"{source_name}_consensus.csv"
    )

    source_consensus.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"✅ {source_name}: "
        f"{len(source_consensus)} satır"
    )

if consensus_results:

    consensus_combined = pd.concat(
        consensus_results,
        ignore_index=True
    )

    combined_output_path = (
        consensus_output_dir
        /
        "all_consensus_combined.csv"
    )

    consensus_combined.to_csv(
        combined_output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"\n✅ Birleşik consensus kaydedildi:"
        f"\n{combined_output_path}"
    )

else:

    print(
        "\n❌ Consensus satırı bulunamadı."
    )